# Detectorv2 speaker-face prompt: MELD dev extraction

Run this notebook with the latest `benchmark-code` and `meld-dataset` Kaggle inputs, Internet, and a GPU. It processes **dev only** and saves frozen Qwen2.5-VL-3B shared embeddings to `/kaggle/working/detectorv2_prompt/meld_dev.pt`. Run train, dev, and test notebooks as separate Kaggle jobs. MLP training is a later step after all three embedding files are available together.

Detectorv2 reads the same sampled frames that Qwen receives. Its action-unit probabilities and blendshape coefficients, scale guide, and per-frame mouth/jaw measurements are added to the video + utterance + emotion task prompt. Face continuity and mouth movement select a *likely* speaker; uncertain clips are marked unavailable.


## 1. Inputs and output paths

Refresh the `benchmark-code` Kaggle dataset from the current repository before running. This notebook uses its extractor directly, as the other MELD experiment notebooks do.


In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
RUN_ROOT = Path("/kaggle/working/detectorv2_prompt")
INDEX_DIR = RUN_ROOT / "indexes"
OUTPUT = RUN_ROOT / "meld_dev.pt"
PREPARE = CODE / "scripts/data/prepare_meld.py"
EXTRACTOR = CODE / "scripts/extraction/extract_meld_detectorv2_prompt.py"
SPLIT = "dev"

for path in (CODE, MELD_ROOT, PREPARE, EXTRACTOR):
    assert path.exists(), f"Missing {path}. Attach current benchmark-code and meld-dataset inputs."
assert "speaker_summary" in EXTRACTOR.read_text() and "detector_cudnn_enabled" in EXTRACTOR.read_text(), (
    "Update the benchmark-code Kaggle input: the Detectorv2 GPU fix is missing.")
INDEX_DIR.mkdir(parents=True, exist_ok=True)
print("Split:", SPLIT, "Output:", OUTPUT)


## 2. Dependencies and GPU

Kaggle's preinstalled Gradio and Diffusers require `huggingface-hub` 1.x. They are unused here; Transformers 4.57.1 requires hub below 1. Extraction starts in a fresh Python process after installation.


In [ ]:
!python -m pip uninstall -y -q gradio diffusers
!python -m pip install -q "py-feat==2.1.5" "transformers==4.57.1" "huggingface-hub>=0.34,<1" "Pillow>=10,<13" "qwen-vl-utils[decord]>=0.0.14" decord accelerate pandas scikit-learn


In [ ]:
from importlib.metadata import version
from packaging.version import Version
import pandas as pd
import torch

assert version("py-feat") == "2.1.5"
assert version("transformers") == "4.57.1"
assert Version("0.34") <= Version(version("huggingface-hub")) < Version("1")
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
print("Py-Feat:", version("py-feat"), "Transformers:", version("transformers"))
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {gpu.name}, {gpu.total_memory / 2**30:.1f} GiB")


## 3. Prepare the dev index

The repository indexer rebuilds video paths against the attached Kaggle dataset and excludes missing clips. This avoids relying on paths saved by an earlier machine.


In [ ]:
!python -u {PREPARE} \
  --meld-root {MELD_ROOT} \
  --split dev \
  --output-dir {INDEX_DIR}

index = pd.read_csv(INDEX_DIR / "meld_dev_index.csv")
assert len(index) > 0 and index.sample_id.is_unique
assert index.split.eq(SPLIT).all() and index.video_exists.eq(True).all()
assert Path(index.video_path.iloc[0]).is_file()
print(SPLIT, len(index), "available clips")


## 4. Extract embeddings

Detectorv2 runs with cuDNN disabled only during its own inference to avoid the GPU convolution-engine failure reported by Py-Feat. Qwen still uses its normal backend. The extractor checkpoints every 25 clips and stops with a traceback on a repeated runtime failure. Rerunning this cell retries previously failed clips. If an older output used different detector settings, it is preserved with a `.pre_cudnn_fix.pt` suffix and a fresh run starts.


In [ ]:
import torch
if OUTPUT.exists():
    previous = torch.load(OUTPUT, map_location="cpu", weights_only=False)
    if previous.get("config", {}).get("detector_cudnn_enabled") is not False:
        backup = OUTPUT.with_suffix(".pre_cudnn_fix.pt")
        assert not backup.exists(), f"Backup already exists: {backup}"
        OUTPUT.rename(backup)
        print("Preserved incompatible prior run:", backup)
    del previous

RESUME = "--resume --retry-errors" if OUTPUT.exists() else ""
!python -u {EXTRACTOR} \
  --index-csv {INDEX_DIR}/meld_dev_index.csv \
  --output-pt {OUTPUT} \
  --fps 6 \
  --min-frames 4 \
  --max-frames 64 \
  --frame-size 224 \
  --face-threshold 0.7 \
  --save-every 25 \
  {RESUME}


## 5. Verify coverage and inspect face selection

The `.pt` file contains embeddings, labels, sample IDs, the full prompt and speaker-face diagnostics for each success, and explicit errors. Review the first prompt and status counts before using the embeddings.


In [ ]:
import json
from collections import Counter
from IPython.display import FileLink, display

payload = torch.load(OUTPUT, map_location="cpu", weights_only=False)
ids = set(payload["sample_ids"])
error_ids = {row["sample_id"] for row in payload["errors"]}
assert len(ids) == len(payload["sample_ids"]) == len(payload["metadata"])
assert not ids & error_ids
assert ids | error_ids == set(index.sample_id), "Some indexed clips were not processed."
assert payload["embeddings"].shape[0] == len(ids) == len(payload["labels"])
assert torch.isfinite(payload["embeddings"]).all()
assert payload["config"]["detector_package"] == "py-feat 2.1.5"
assert payload["config"]["detector_cudnn_enabled"] is False
print("Embeddings:", tuple(payload["embeddings"].shape))
print("Errors:", len(payload["errors"]), "First errors:", payload["errors"][:5])
print("Face status:", Counter(row["speaker_face"]["status"] for row in payload["metadata"]))
if payload["metadata"]:
    print("Example prompt:\n", payload["metadata"][0]["prompt"][:2500])
(RUN_ROOT / f"meld_{SPLIT}_errors.json").write_text(json.dumps(payload["errors"], indent=2))
display(FileLink(str(OUTPUT)))
